In [3]:
import json
import faiss
import numpy as np
import requests
from sentence_transformers import SentenceTransformer

In [4]:
# Load previously created document chunks

with open("../data&doc/chunks.json", "r", encoding="utf-8") as file:
    chunks = json.load(file)

print("Total chunks:", len(chunks))
print("\nFirst chunk:")
print(chunks[0])

Total chunks: 5

First chunk:
ABC Technologies | Internal Sample Document Page 1 ABC TECHNOLOGIES EMPLOYEE LEAVE POLICY Document ID HR-POL-001 Version 1.0 Effective Date 01 January 2026 Department Human Resources 1. Purpose This policy explains the leave benefits and request process for employees of ABC Technologies. It is intended to help employees understand the available leave types and the steps required to request time off. 2. Annual Leave Eligible full-time employees receive 18 days of annual leave per calendar year.


In [5]:
# Load previously saved FAISS index

index = faiss.read_index("../data&doc/faiss_index.bin")

print("FAISS index loaded successfully!")
print("Total vectors:", index.ntotal)

FAISS index loaded successfully!
Total vectors: 5


In [7]:
# Load the same embedding model used in Semantic Search

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5196.03it/s]


Embedding model loaded successfully!


In [21]:
# User question

question = question = "What is the maternity leave policy?"
print("User Question:", question)

User Question: What is the maternity leave policy?


In [22]:
# Convert user question into vector

question_embedding = embedding_model.encode([question])

# Convert into FAISS-compatible format
question_embedding = np.array(question_embedding).astype("float32")

print("Question embedding shape:", question_embedding.shape)

Question embedding shape: (1, 384)


In [23]:
# Search top 2 relevant chunks

distances, indices = index.search(question_embedding, k=2)

print("Retrieved Document Chunks:\n")

for rank, idx in enumerate(indices[0], start=1):

    print(f"Chunk {rank}:")
    print(chunks[idx])
    print("-" * 50)

Retrieved Document Chunks:

Chunk 1:
ABC Technologies | Internal Sample Document Page 1 ABC TECHNOLOGIES EMPLOYEE LEAVE POLICY Document ID HR-POL-001 Version 1.0 Effective Date 01 January 2026 Department Human Resources 1. Purpose This policy explains the leave benefits and request process for employees of ABC Technologies. It is intended to help employees understand the available leave types and the steps required to request time off. 2. Annual Leave Eligible full-time employees receive 18 days of annual leave per calendar year.
--------------------------------------------------
Chunk 2:
before making non-refundable travel arrangements. 6. Unplanned Absence For an unexpected absence, employees should notify their manager as early as possible and submit the relevant request in the HR portal when they are able to do so. 7. Questions For questions about leave balances, eligibility, or this policy, employees should contact the Human Resources department at hr@abctech.example. ABC Technolo

In [24]:
# Combine retrieved chunks into context

retrieved_context = ""

for idx in indices[0]:
    retrieved_context += chunks[idx] + "\n\n"

print("Retrieved Context:")
print(retrieved_context)

Retrieved Context:
ABC Technologies | Internal Sample Document Page 1 ABC TECHNOLOGIES EMPLOYEE LEAVE POLICY Document ID HR-POL-001 Version 1.0 Effective Date 01 January 2026 Department Human Resources 1. Purpose This policy explains the leave benefits and request process for employees of ABC Technologies. It is intended to help employees understand the available leave types and the steps required to request time off. 2. Annual Leave Eligible full-time employees receive 18 days of annual leave per calendar year.

before making non-refundable travel arrangements. 6. Unplanned Absence For an unexpected absence, employees should notify their manager as early as possible and submit the relevant request in the HR portal when they are able to do so. 7. Questions For questions about leave balances, eligibility, or this policy, employees should contact the Human Resources department at hr@abctech.example. ABC Technologies | Internal Sample Document Page 2 Note: ABC Technologies is a fictional 

In [25]:
# Create a prompt using retrieved context

prompt = f""" You are a company document assistant.

Answer the user's question using ONLY the
provided company document context.

If the answer is not available in the context,
say "The information is not available in the document."

Keep the answer short and simple.
Use maximum 3 bullet points.

Company Document Context:
{retrieved_context}

User Question:
{question}

Answer:
"""

# Send prompt to Ollama

response = requests.post("http://localhost:11434/api/generate",json={"model": "qwen2.5:3b", "prompt": prompt, "stream": False, 
"options": {"temperature": 0.2,"num_predict": 120}})

result = response.json()

print("Final AI Answer:")
print(result["response"])

Final AI Answer:
The information is not available in the document.
